# Assignment 4 Report: Sharding of the Developed Application
 
 **Project:** FixIIT Complaint Management System  
**Group Name:** DataSquad 

**Github_Link:** https://github.com/Vishawraj388/CS432-track1_assignment4 

**Video_Link:** https://youtu.be/gxYv0LgKUYU


## 1. Overview

The objective of this assignment is to extend the existing database backed application with horizontal scaling behavior through logical sharding. Instead of keeping every complaint only in one table, complaint records are divided into one of three shard tables based on a shard key. Application logic is added which can then decides which shard should be queried for lookups, inserts, updates, deletes, and filtered complaint listings.

## 2. Shard Key Selection

The selected shard key is:

```text
member_id
```

Complaints are naturally owned by members, and each complaint row already contains `member_id`. This makes it a suitable key for routing complaint-related operations.

### Justification

**High cardinality:** The `member_id` column has many possible values because each user can create many complaints. This distributes complaint rows better than low cardinality fields such as `status_id` or `priority_id`.

**Query aligned:** Complaint APIs frequently need member specific access. Normal users should only see their own complaints, and staff/admin users can filter complaints by member. Since `member_id` appears repeatedly for these accesses, we find it useful for routing. We do not want unwanted access by unauthorized users. 

**Stable:** Once a complaint is created, its owning member does not normally change. Complaints can only be created by a member with his own `member_id`. This avoids expensive record movement between shards after insertion.

## 3. Partitioning Strategy

The implementation uses hash based partitioning with 3 shards:

```text
shard_id =member_id % 3
```

The 3 shard tables are:
- `shard_0_complaint`
- `shard_1_complaint`
- `shard_2_complaint`

An intermediate table called `complaint_shard_map` stores the mapping between each `complaint_id`, its `member_id`, and the shard where the complaint is stored.

Hash-based sharding was chosen because it is simple and predictable, and gives a more equal distribution than range based sharding for this dataset. Range based sharding could become skewed if most new complaint IDs or members fall into the same range this could be an issue if all complaints coming at a time are of same type. The modulo-based strategy ensures that each shard has equal number of members based on their ID value.

## 4. Shard Tables Created

The original `complaint` table is converted to a 3 new sharded complaint tables. The database was extended with the following sharding objects:

```sql
CREATE TABLE complaint_shard_map (
    complaint_id INTEGER PRIMARY KEY,
    member_id    INTEGER NOT NULL,
    shard_id     INTEGER NOT NULL CHECK (shard_id >= 0 AND shard_id < 3)
);
```

Each shard table has the complaint columns needed by the application which are the same as unsharded design:

```text
complaint_id, member_id, issue_type_id, priority_id, status_id, description, created_at, closed_at, hostel_id, hostel_room_no, location_id
```

Indexes were also added on shard columns used for routing and filtering, especially `member_id` and `created_at` which will be commonly used.

## 5. Data Migration

Existing complaint rows were migrated from the original `complaint` table into the shard tables using the rule `member_id % 3`.

The migration logic is equivalent to:

```sql
INSERT INTO shard_0_complaint
SELECT * FROM complaint WHERE member_id % 3 = 0;

INSERT INTO shard_1_complaint
SELECT * FROM complaint WHERE member_id % 3 = 1;

INSERT INTO shard_2_complaint
SELECT * FROM complaint WHERE member_id % 3 = 2;

INSERT INTO complaint_shard_map (complaint_id, member_id, shard_id)
SELECT complaint_id, member_id, member_id % 3
FROM complaint;
```

After migration, each complaint should appear in exactly one shard. The mapping table allows direct lookup of the shard for a known complaint ID.

In [ ]:
import sqlite3
from pathlib import Path

db_path = Path('app/local_database.db')
conn = sqlite3.connect(db_path)
conn.row_factory = sqlite3.Row

tables = ['complaint', 'complaint_shard_map', 'shard_0_complaint', 'shard_1_complaint', 'shard_2_complaint']
for table in tables:
    total = conn.execute(f'SELECT COUNT(*) AS total FROM {table}').fetchone()['total']
    print(f'{table}: {total}')

conn.close()

Expected verified result from the current database:

```text
complaint: 16
complaint_shard_map: 16
shard_0_complaint: 5
shard_1_complaint: 5
shard_2_complaint: 6
```

The base complaint table has 16 records, the shard map has 16 records, and the combined shard total is also 16. This confirms that no complaints were lost during migration.

## 6. Query Routing Implementation

The application code in `app/app.py` was updated to route complaint queries through shard helper functions.

Important routing functions include:

- `get_complaint_shard_id(member_id)`: calculates `member_id % 3` required for finding correct shard
- `complaint_shard_table(shard_id)`: returns the shard table name from shard_id
- `resolve_target_shards(current_user, member_id_filter)`: decides which shards a request must query 
- `fetch_sharded_complaints()`: queries one or more (if required) shard tables and merges the result
- `get_shard_mapping(complaint_id)`: finds the shard for a complaint ID using `complaint_shard_map`
- `fetch_sharded_complaint_by_id(complaint_id)`: performs a single complaint lookup against the correct shard

### Insert Routing

When a new complaint is created, the app inserts the row into the base `complaint` table, calculates the shard using the complaint owner's `member_id`, inserts the same complaint into the correct shard table, and records the mapping in `complaint_shard_map`.

### Lookup Routing

For a request such as `/complaints/<complaint_id>`, the app first checks `complaint_shard_map` to find the target shard. It then queries only that shard table.

### Range and Filtered Queries

For listing complaints, the app identifies the required shards. If a member filter is present, only one shard is queried. If an admin or staff user requests all complaints or a date/status range, all three shards may be queried and the results are merged in application code. This is a scatter-gather pattern.

## 7. Verification Results

The verification script `verify_sharding.py` checks that the sharding result is correct.

Observed result:

```text
Shard 0: table=shard_0_complaint, complaints=5, distinct_members=5
Shard 1: table=shard_1_complaint, complaints=5, distinct_members=5
Shard 2: table=shard_2_complaint, complaints=6, distinct_members=6

Base complaint count: 16
Shard map count:      16
Shard total count:    16

Verification result: counts match across base table, map, and shards.
Duplicate records across shards: 0
Rows in wrong shard:            0
```

This verifies three important properties:

- Every original complaint exists in a shard.
- No complaint appears in more than one shard.
- Every complaint is stored in the shard required by `member_id % 3`.

In [ ]:
import sqlite3
from pathlib import Path

db_path = Path('app/local_database.db')
conn = sqlite3.connect(db_path)
conn.row_factory = sqlite3.Row

duplicate_total = conn.execute('''
SELECT COUNT(*) AS total
FROM (
    SELECT complaint_id
    FROM (
        SELECT complaint_id FROM shard_0_complaint
        UNION ALL
        SELECT complaint_id FROM shard_1_complaint
        UNION ALL
        SELECT complaint_id FROM shard_2_complaint
    )
    GROUP BY complaint_id
    HAVING COUNT(*) > 1
)
''').fetchone()['total']

wrong_shard_total = 0
for shard_id in range(3):
    wrong_shard_total += conn.execute(
        f'SELECT COUNT(*) AS total FROM shard_{shard_id}_complaint WHERE member_id % 3 <> ?',
        (shard_id,)
    ).fetchone()['total']

print('Duplicate records across shards:', duplicate_total)
print('Rows in wrong shard:', wrong_shard_total)

conn.close()

## 8. Scalability and Trade offs Analysis

### Horizontal vs Vertical Scaling

Vertical scaling improves one database server by adding more CPU, memory or storage, this is fine but sharding is more optimal. Sharding represents horizontal scaling because data is sharded across partitions that can eventually stay on separate machines. This allows the system to grow by adding more shards rather than only upgrading one server. This also stops failure at one server to stop entire functionality.

### Consistency

In this implementation, the base table and shard tables are updated in the same SQLite transaction for inserts, updates, and deletes. This keeps them consistent locally. In a distributed system, consistency becomes harder because different shards may be on different machines. A network failure or partial write could cause one shard to be updated while another related operation fails. In our implementation consistency with shards is not an issue and all shards can run upto date data.

### Availability

If one shard goes down in a sharded system, only data on that shard becomes unavailable. Other shards can continue serving requests. This improves partial availability compared with a single shard database, where one database failure can stop the entire application. We need this kind of splitting to avoid complete break down of the system.

### Partition Tolerance

Partition tolerance means the system should continue behaving predictably even when communication between nodes fails. Since this assignment simulates shards inside one SQLite database, true network partitions are simulated. The design shows how failure could be isolated: if `shard_1` failed, requests for members where `member_id % 3 = 1` would fail, while requests for shard 0 and shard 2 could still succeed.

### Scatter Gather Cost

Queries that do not specify a shard key may need to query all shards and merge the results. This is slower than a single-shard lookup. For example, an admin viewing all complaints or filtering by a date range may require querying `shard_0_complaint`, `shard_1_complaint`, and `shard_2_complaint`. This may be slower than single shard design based on the query. 

## 9. Observations and Limitations

The sharding implementation successfully partitions complaint data across three shard tables and routes application operations based on `member_id`. Verification confirms that no complaint records were lost or duplicated.

In this project, shards were simulated inside a single SQLite database file (`app/local_database.db`) by creating three separate complaint shard tables: `shard_0_complaint`, `shard_1_complaint`, and `shard_2_complaint`. Each complaint is routed to exactly one of these tables using the rule `member_id % 3`, and the `complaint_shard_map` table records which shard contains each complaint. Therefore, this implementation counts as logical table-based shard isolation within one database, not physical shard isolation using separate databases, servers, or containers.


The main limitation is that this is a local simulation. All shards are stored in the same SQLite database file, so the project does not demonstrate isolation, independent shard failures, replication, or distributed transactions. Another limitation is that the base `complaint` table is still retained as a source of truth, while a fully sharded production design might store complaint data only in shard nodes with separate metadata services. This is used however only for the sake of comparison.

Despite these limitations, the implementation demonstrates the core ideas required for the assignment: shard key selection, logical partitioning, data migration, query routing, verification, and trade-off analysis.

## 10. Conclusion

The FixIIT application was extended with logical sharding for complaint records. The selected shard key is `member_id`, and the partitioning strategy is hash based using `member_id % 3`. Three shard tables store complaint partitions, and a mapping table supports direct complaint lookup routing. Application logic routes inserts, lookups, updates, deletes, and filtered listing queries through the sharded structure.

The final verification confirms that the base table, shard map, and shard tables all contain matching complaint counts, with no duplicate records and no rows placed in the wrong shard.